In [0]:
import os
import pickle
import re
import pandas as pd

bbox_path = "/Volumes/workspace/default/ucf_crime/annotations/bounding-boxes"

train_pkl_path = os.path.join(bbox_path, "Train_annotation.pkl")
test_pkl_path = os.path.join(bbox_path, "Test_annotation.pkl")

In [0]:
with open(train_pkl_path, "rb") as f:
    train_annotations = pickle.load(f, encoding="latin1")

with open(test_pkl_path, "rb") as f:
    test_annotations = pickle.load(f, encoding="latin1")

print("Videos in train file:", len(train_annotations))
print("Videos in test file:", len(test_annotations))


In [0]:
def extract_frame_number(frame_filename): 
    match = re.search(r"(\d+)", frame_filename)
    return int(match.group(1)) if match else None

def extract_category(video_name):
    match = re.match(r"^([A-Za-z]+)", video_name)
    return match.group(1) if match else None

def flatten_annotations(annotation_dict, split_name):
    rows = []
    for video_name, frame_list in annotation_dict.items():
        for frame_entry in frame_list:
            frame_path = frame_entry[0]
            x1, y1, x2, y2 = frame_entry[1], frame_entry[2], frame_entry[3], frame_entry[4]
            frame_filename = os.path.basename(frame_path)

            rows.append({
                "split": split_name,
                "video_name": video_name,
                "category": extract_category(video_name),
                "frame_path": frame_path,
                "frame_number": extract_frame_number(frame_filename),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
            })
    return rows

In [0]:
train_rows = flatten_annotations(train_annotations, "train")
test_rows = flatten_annotations(test_annotations, "test")

annotations_df = pd.DataFrame(train_rows + test_rows)

print("Total rows (one per annotated frame):", len(annotations_df))
print(annotations_df.shape)
annotations_df.head(10)



In [0]:
annotations_df["box_width"] = annotations_df["x2"] - annotations_df["x1"]
annotations_df["box_height"] = annotations_df["y2"] - annotations_df["y1"]
annotations_df["box_area"] = annotations_df["box_width"] * annotations_df["box_height"]

annotations_df.head(50)

In [0]:
print("Rows per split:")
print(annotations_df["split"].value_counts())

print("\nRows per category (top 15):")
print(annotations_df["category"].value_counts().head(15))

print("\nAny null values?")
print(annotations_df.isnull().sum())

print("\nBox coordinate ranges:")
print(annotations_df[["x1", "y1", "x2", "y2", "box_width", "box_height"]].describe())

In [0]:
output_csv_path = "/Volumes/workspace/default/ucf_crime/annotations/bounding-boxes/annotations_flat.csv"
annotations_df.to_csv(output_csv_path, index=False)
print("Saved combined annotation dataframe to:", output_csv_path)